# Credit Card Fraud: Final Test Evaluation

This notebook reproduces the frozen August evaluation on the external `fraud_test.csv` dataset. It freezes the selected XGBoost pipeline, score cutoffs, customer-history boundaries, contextual flags, and ordered decision rules established using `fraud_train.csv`.

The test target is not used for model fitting, preprocessing, threshold selection, behavioral-boundary selection, or rule design. Scores and rule decisions are produced first; `is_fraud` is then used only to measure final performance.

The main results reproduce the August baseline. The appendix replays its decisions with a normalized customer-history key.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score, ConfusionMatrixDisplay,
    PrecisionRecallDisplay, RocCurveDisplay,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda value: f'{value:,.6f}')
plt.style.use('seaborn-v0_8-whitegrid')

RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd()
TRAIN_PATH = PROJECT_ROOT / 'Data' / 'raw' / 'fraud_train.csv'
TEST_PATH = PROJECT_ROOT / 'Data' / 'raw' / 'fraud_test.csv'
OUTPUT_DIR = PROJECT_ROOT / 'Data' / 'processed'

for path in [TRAIN_PATH, TEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'Missing required data file: {path}')

### Frozen model and cutoffs
The model settings, score cutoffs and rule hierarchy below were selected using the training and validation periods. They are fixed for the holdout replay.

In [ ]:
# Frozen score thresholds from the latest-30% training validation period.
T_HARD = 0.9620440006256104
T_MEDIUM = 0.7880805730819702
T_LOW = 0.14788001775741577

# Frozen customer-history boundaries learned from the earliest-70% development period.
LIMITED_HISTORY_MAX = 99
RAPID_REPEAT_MAX_MINUTES = 33.05
LONG_INACTIVITY_MIN_MINUTES = 1254.18
AMOUNT_SPIKE_MIN_RATIO = 2.055752432258314

HIGH_RISK_CATEGORIES = {'shopping_net', 'misc_net', 'grocery_pos'}
HIGH_RISK_HOURS = {22, 23, 0, 1, 2, 3}

frozen_parameters = pd.Series({
    'T_HARD': T_HARD,
    'T_MEDIUM': T_MEDIUM,
    'T_LOW': T_LOW,
    'LIMITED_HISTORY_MAX': LIMITED_HISTORY_MAX,
    'RAPID_REPEAT_MAX_MINUTES': RAPID_REPEAT_MAX_MINUTES,
    'LONG_INACTIVITY_MIN_MINUTES': LONG_INACTIVITY_MIN_MINUTES,
    'AMOUNT_SPIKE_MIN_RATIO': AMOUNT_SPIKE_MIN_RATIO,
})
display(frozen_parameters.to_frame('frozen_value'))

## 2. Feature engineering and data loading

The model features exactly match notebook 02. Customer-history features are calculated over training and test transactions in chronological order, with source and row order breaking timestamp ties. Within the same recorded minute, true event order in the historical export can be uncertain. Test labels are never used when constructing history.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    delta_lat = lat2 - lat1
    delta_lon = lon2 - lon1
    a = (
        np.sin(delta_lat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(delta_lon / 2) ** 2
    )
    return 2 * 6371.0088 * np.arcsin(np.sqrt(a))


def prepare_model_features(raw):
    data = raw.drop(
        columns=[column for column in raw.columns if column.startswith('Unnamed:')],
        errors='ignore',
    ).copy()
    transaction_time = pd.to_datetime(
        data['trans_date_trans_time'], format='mixed', dayfirst=True, errors='coerce'
    )
    birth_date = pd.to_datetime(
        data['dob'], format='mixed', dayfirst=True, errors='coerce'
    )

    features = pd.DataFrame(index=data.index)
    features['amt_log1p'] = np.log1p(data['amt'])
    features['city_pop_log1p'] = np.log1p(data['city_pop'])
    features['age_at_transaction'] = (transaction_time - birth_date).dt.days / 365.2425
    features['trans_hour'] = transaction_time.dt.hour
    features['trans_day_of_week'] = transaction_time.dt.dayofweek
    features['trans_month'] = transaction_time.dt.month
    features['is_weekend'] = transaction_time.dt.dayofweek.ge(5).astype('int8')
    features['is_night'] = transaction_time.dt.hour.between(0, 5).astype('int8')
    features['hour_sin'] = np.sin(2 * np.pi * transaction_time.dt.hour / 24)
    features['hour_cos'] = np.cos(2 * np.pi * transaction_time.dt.hour / 24)
    features['day_sin'] = np.sin(2 * np.pi * transaction_time.dt.dayofweek / 7)
    features['day_cos'] = np.cos(2 * np.pi * transaction_time.dt.dayofweek / 7)
    features['merchant_distance_km'] = haversine_km(
        data['lat'].to_numpy(), data['long'].to_numpy(),
        data['merch_lat'].to_numpy(), data['merch_long'].to_numpy(),
    )
    features['lat'] = data['lat']
    features['long'] = data['long']
    for column in ['category', 'gender', 'state']:
        features[column] = data[column].astype('string').str.strip().fillna('missing')
    return features, transaction_time


def prepare_combined_history(train_raw, test_raw):
    train_history = train_raw.copy()
    test_history = test_raw.copy()
    train_history['_source'] = 'train'
    test_history['_source'] = 'test'
    train_history['_source_rank'] = 0
    test_history['_source_rank'] = 1
    train_history['_source_row'] = np.arange(len(train_history))
    test_history['_source_row'] = np.arange(len(test_history))
    combined = pd.concat([train_history, test_history], ignore_index=True)
    combined['_transaction_time'] = pd.to_datetime(
        combined['trans_date_trans_time'], format='mixed', dayfirst=True, errors='coerce'
    )
    combined['_merchant_normalized'] = (
        combined['merchant'].astype('string').str.removeprefix('fraud_').str.strip()
    )
    profile_columns = ['first', 'last', 'dob', 'street']
    combined['_customer_key'] = pd.util.hash_pandas_object(
        combined[profile_columns].astype('string'), index=False
    ).astype('uint64')

    combined = combined.sort_values(
        ['_customer_key', '_transaction_time', '_source_rank', '_source_row'],
        kind='stable',
    ).copy()
    customer_group = combined.groupby('_customer_key', sort=False)
    combined['customer_prior_txn_count'] = customer_group.cumcount()
    combined['minutes_since_previous_txn'] = (
        customer_group['_transaction_time'].diff().dt.total_seconds().div(60)
    )
    combined['is_first_observed_txn'] = combined[
        'minutes_since_previous_txn'
    ].isna()

    prior_amount_sum = customer_group['amt'].cumsum() - combined['amt']
    prior_amount_mean = prior_amount_sum.div(
        combined['customer_prior_txn_count'].replace(0, np.nan)
    )
    combined['amount_to_prior_mean'] = (
        combined['amt'] / prior_amount_mean.replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan)
    combined['customer_merchant_prior_count'] = combined.groupby(
        ['_customer_key', '_merchant_normalized'], sort=False
    ).cumcount()
    combined['is_new_merchant_for_customer'] = combined[
        'customer_merchant_prior_count'
    ].eq(0)

    history_columns = [
        'customer_prior_txn_count', 'minutes_since_previous_txn',
        'is_first_observed_txn', 'amount_to_prior_mean',
        'customer_merchant_prior_count', 'is_new_merchant_for_customer',
    ]
    return (
        combined.loc[combined['_source'].eq('test')]
        .sort_values('_source_row', kind='stable')[history_columns]
        .reset_index(drop=True)
    )


identifier_types = {'cc_num': 'string', 'zip': 'string', 'trans_num': 'string'}
train_raw = pd.read_csv(TRAIN_PATH, dtype=identifier_types, low_memory=False)
test_raw = pd.read_csv(TEST_PATH, dtype=identifier_types, low_memory=False)

X_train_all, train_time = prepare_model_features(train_raw)
X_test, test_time = prepare_model_features(test_raw)
test_history = prepare_combined_history(train_raw, test_raw)

print(f'Official training rows: {len(train_raw):,}')
print(f'External test rows: {len(test_raw):,}')
print(f'Training dates: {train_time.min()} to {train_time.max()}')
print(f'Test dates: {test_time.min()} to {test_time.max()}')

The historical export uses a different birth-date format in each input. The baseline uses the original text key, so returning profiles start with an empty history at the split. The final diagnostic measures the effect of harmonizing this field.

## 3. Fit the frozen XGBoost model and score the test set

The model remains fitted on the earliest 70% of `fraud_train.csv`, exactly as it was when the score thresholds were established. The latest 30% training validation period is not added to fitting because doing so would change the model's score scale and invalidate the frozen cutoffs.

In [ ]:
train_order = np.argsort(train_time.to_numpy(), kind='stable')
X_train_all = X_train_all.iloc[train_order].reset_index(drop=True)
y_train_all = train_raw['is_fraud'].astype('int8').iloc[train_order].reset_index(drop=True)
development_end = int(len(X_train_all) * 0.70)
X_development = X_train_all.iloc[:development_end]
y_development = y_train_all.iloc[:development_end]

numeric_features = [
    'amt_log1p', 'city_pop_log1p', 'age_at_transaction', 'trans_hour',
    'trans_day_of_week', 'trans_month', 'is_weekend', 'is_night',
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    'merchant_distance_km', 'lat', 'long',
]
categorical_features = ['category', 'gender', 'state']

preprocessor = ColumnTransformer([
    ('numeric', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), numeric_features),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('one_hot', OneHotEncoder(handle_unknown='ignore', min_frequency=20)),
    ]), categorical_features),
])

scale_pos_weight = y_development.eq(0).sum() / y_development.eq(1).sum()
frozen_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(
        n_estimators=117, learning_rate=0.05, max_depth=8,
        min_child_weight=10, gamma=0.1, reg_lambda=5.0,
        subsample=0.9, colsample_bytree=0.9,
        scale_pos_weight=scale_pos_weight,
        objective='binary:logistic', eval_metric='aucpr', tree_method='hist',
        n_jobs=-1, random_state=RANDOM_STATE,
    )),
])

frozen_model.fit(X_development, y_development)
test_score = frozen_model.predict_proba(X_test)[:, 1]

assert len(test_score) == len(test_raw)
assert np.isfinite(test_score).all()
print(f'Scored {len(test_score):,} external test transactions.')
print('Test labels have not been used in model fitting or score generation.')

**What this block does:** Reconstructs and fits the exact selected XGBoost pipeline on the frozen development period, then generates one risk score for every external test transaction without using the test target.

## 4. Generate frozen behavioral flags and ordered rule decisions

The rules are applied in first-hit order. Mandatory policy and verified-whitelist rules are not evaluated because the dataset lacks the required fields. Consequently, this notebook evaluates only the model-and-behavior portion of the proposed rule engine.

In [ ]:
def apply_policy(test_score, amounts, X_test, test_history):
    decision_frame = pd.DataFrame({
        'score': test_score,
        'amount': amounts,
        'category': X_test['category'].to_numpy(),
        'transaction_hour': X_test['trans_hour'].to_numpy(),
    })
    decision_frame = pd.concat([decision_frame, test_history], axis=1)

    decision_frame['first_observed'] = decision_frame['is_first_observed_txn']
    decision_frame['limited_history'] = (
        ~decision_frame['first_observed']
        & decision_frame['customer_prior_txn_count'].le(LIMITED_HISTORY_MAX)
    )
    decision_frame['rapid_repeat'] = (
        ~decision_frame['first_observed']
        & decision_frame['minutes_since_previous_txn'].le(RAPID_REPEAT_MAX_MINUTES)
    )
    decision_frame['long_inactivity'] = (
        ~decision_frame['first_observed']
        & decision_frame['minutes_since_previous_txn'].ge(LONG_INACTIVITY_MIN_MINUTES)
    )
    decision_frame['amount_spike'] = (
        ~decision_frame['first_observed']
        & decision_frame['amount_to_prior_mean'].ge(AMOUNT_SPIKE_MIN_RATIO)
    )
    decision_frame['new_merchant'] = decision_frame['is_new_merchant_for_customer']
    decision_frame['high_risk_category'] = decision_frame['category'].isin(HIGH_RISK_CATEGORIES)
    decision_frame['high_risk_time'] = decision_frame['transaction_hour'].isin(HIGH_RISK_HOURS)

    flag_columns = [
        'first_observed', 'limited_history', 'rapid_repeat', 'long_inactivity',
        'amount_spike', 'new_merchant', 'high_risk_category', 'high_risk_time',
    ]
    decision_frame['behavior_flag_count'] = (
        decision_frame[flag_columns].astype('int8').sum(axis=1)
    )

    hard_combined = (
        decision_frame['score'].ge(T_HARD)
        & decision_frame['behavior_flag_count'].ge(3)
    )
    hard_score_soft = decision_frame['score'].ge(T_HARD) & ~hard_combined
    medium_band = decision_frame['score'].ge(T_MEDIUM) & decision_frame['score'].lt(T_HARD)
    history_context = (
        medium_band
        & (decision_frame['first_observed'] | decision_frame['limited_history'])
        & (decision_frame['high_risk_category'] | decision_frame['high_risk_time'])
    )
    new_merchant_spike = (
        medium_band & decision_frame['new_merchant'] & decision_frame['amount_spike']
    )
    interval_context = (
        medium_band
        & (decision_frame['rapid_repeat'] | decision_frame['long_inactivity'])
        & (decision_frame['high_risk_category'] | decision_frame['high_risk_time'])
    )

    conditions = [
        hard_combined,
        hard_score_soft,
        history_context,
        new_merchant_spike,
        interval_context,
        medium_band,
        decision_frame['score'].lt(T_LOW),
    ]
    rule_names = [
        'CC1.1 Hard decline: extreme score + >=3 flags',
        'CC1.2 Soft decline: extreme score',
        'CC1.3 Soft decline: history uncertainty + risky context',
        'CC1.4 Soft decline: new merchant + amount spike',
        'CC1.5 Soft decline: interval anomaly + risky context',
        'CC1.6 Soft decline: remaining medium score',
        'CS1.1 Soft approve: score below T_low',
    ]
    decision_frame['triggered_rule'] = np.select(
        conditions, rule_names,
        default='CS1.2 Soft approve: lower-score activity',
    )
    decision_frame['decision'] = np.select(
        [hard_combined, hard_score_soft | medium_band],
        ['Hard Decline', 'Soft Decline'],
        default='Soft Approve',
    )

    assert decision_frame['triggered_rule'].notna().all()
    assert decision_frame['decision'].notna().all()
    return decision_frame


decision_frame = apply_policy(test_score, test_raw['amt'].to_numpy(), X_test, test_history)
display(decision_frame['decision'].value_counts().to_frame('transactions'))
display(decision_frame['triggered_rule'].value_counts().to_frame('transactions'))

**What this block does:** Computes the eight frozen customer-history/context flags and applies the ordered rules. Each transaction receives exactly one first-hit rule and one operational decision before the target is consulted.

## 5. Reveal the test target and measure final performance

Only at this stage is `is_fraud` attached to the already-scored and already-decided transactions. No thresholds or rule conditions are changed after viewing these results.

In [ ]:
y_test = test_raw['is_fraud'].astype('int8').reset_index(drop=True)
decision_frame['is_fraud'] = y_test
decision_frame['transaction_amount'] = test_raw['amt'].to_numpy()

model_metrics = pd.DataFrame([{
    'evaluation_dataset': 'External fraud_test.csv',
    'rows': len(y_test),
    'fraud_cases': int(y_test.sum()),
    'fraud_rate': y_test.mean(),
    'ROC-AUC': roc_auc_score(y_test, test_score),
    'PR-AUC': average_precision_score(y_test, test_score),
}])

threshold_rows = []
for name, threshold in [('T_hard', T_HARD), ('T_medium', T_MEDIUM), ('T_low', T_LOW)]:
    prediction = test_score >= threshold
    tn, fp, fn, tp = confusion_matrix(y_test, prediction).ravel()
    threshold_rows.append({
        'threshold_name': name,
        'threshold': threshold,
        'flagged_transactions': int(prediction.sum()),
        'flag_rate': prediction.mean(),
        'true_positives': int(tp),
        'false_positives': int(fp),
        'false_negatives': int(fn),
        'precision': precision_score(y_test, prediction, zero_division=0),
        'recall': recall_score(y_test, prediction, zero_division=0),
        'f1': f1_score(y_test, prediction, zero_division=0),
    })
threshold_results = pd.DataFrame(threshold_rows)

rule_results = (
    decision_frame.groupby(['triggered_rule', 'decision'], observed=True)
    .agg(
        transactions=('is_fraud', 'size'),
        fraud_cases=('is_fraud', 'sum'),
        transaction_amount=('transaction_amount', 'sum'),
        fraud_amount=('transaction_amount', lambda values: values[decision_frame.loc[values.index, 'is_fraud'].eq(1)].sum()),
    )
    .reset_index()
)
rule_results['good_transactions'] = rule_results['transactions'] - rule_results['fraud_cases']
rule_results['fraud_hit_rate'] = rule_results['fraud_cases'] / rule_results['transactions']
rule_results['share_of_all_fraud'] = rule_results['fraud_cases'] / int(y_test.sum())
rule_results = rule_results.sort_values('fraud_cases', ascending=False).reset_index(drop=True)

policy_prediction = decision_frame['decision'].isin(['Hard Decline', 'Soft Decline'])
tn, fp, fn, tp = confusion_matrix(y_test, policy_prediction).ravel()
policy_metrics = pd.DataFrame([{
    'policy': 'Frozen model-and-behavior rules',
    'declined_or_verified': int(policy_prediction.sum()),
    'decision_rate': policy_prediction.mean(),
    'true_positives': int(tp),
    'false_positives': int(fp),
    'false_negatives': int(fn),
    'true_negatives': int(tn),
    'precision': precision_score(y_test, policy_prediction, zero_division=0),
    'recall': recall_score(y_test, policy_prediction, zero_division=0),
    'f1': f1_score(y_test, policy_prediction, zero_division=0),
}])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model_metrics.to_csv(OUTPUT_DIR / 'final_test_model_metrics.csv', index=False)
threshold_results.to_csv(OUTPUT_DIR / 'final_test_threshold_results.csv', index=False)
rule_results.to_csv(OUTPUT_DIR / 'final_test_rule_results.csv', index=False)
policy_metrics.to_csv(OUTPUT_DIR / 'final_test_policy_metrics.csv', index=False)

display(model_metrics)
display(threshold_results)
display(policy_metrics)
display(rule_results)

**What this block does:** Reveals the external test labels and calculates final ranking, cutoff, and first-hit rule performance. It saves aggregate results only; no row-level identifiers or test decisions are exported.

## 6. Final visual evaluation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)

RocCurveDisplay.from_predictions(y_test, test_score, ax=axes[0], name='Frozen XGBoost')
axes[0].set_title('External Test ROC Curve')

PrecisionRecallDisplay.from_predictions(y_test, test_score, ax=axes[1], name='Frozen XGBoost')
axes[1].axhline(y_test.mean(), color='gray', linestyle='--', label='Fraud-rate baseline')
axes[1].set_title('External Test Precision–Recall Curve')
axes[1].legend()

ConfusionMatrixDisplay.from_predictions(
    y_test, policy_prediction.astype('int8'),
    display_labels=['Approve', 'Decline/Verify'],
    cmap='Blues', values_format=',d', ax=axes[2],
)
axes[2].set_yticklabels(['Legitimate', 'Fraud'])
axes[2].set_ylabel('Observed transaction')
axes[2].set_xlabel('Policy action')
axes[2].set_title('Frozen Policy Confusion Matrix')
(PROJECT_ROOT / 'figures').mkdir(exist_ok=True)
fig.savefig(PROJECT_ROOT / 'figures' / 'holdout-performance.png', dpi=150, bbox_inches='tight')
plt.show()

plot_rules = rule_results.sort_values('fraud_hit_rate')
fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
axes[0].barh(plot_rules['triggered_rule'], plot_rules['fraud_hit_rate'].mul(100), color='#E15759')
axes[0].set_title('External Test Fraud Hit Rate by First-Hit Rule')
axes[0].set_xlabel('Fraud hit rate (%)')

axes[1].barh(plot_rules['triggered_rule'], plot_rules['transactions'], color='#4C78A8')
axes[1].set_xscale('log')
axes[1].set_title('External Test Transaction Volume by Rule')
axes[1].set_xlabel('Transactions (log scale)')
plt.show()

**What this block does:** Visualizes the final test ranking quality, policy confusion matrix, and the hit-rate/volume trade-off for every first-hit rule. The log-volume chart keeps small high-risk rules visible beside the much larger approve population.

## 7. Specific final-test conclusion

In [ ]:
validation_pr_auc = 0.9145147364296071
test_pr_auc = float(model_metrics.loc[0, 'PR-AUC'])

validation_threshold_path = OUTPUT_DIR / 'xgboost_threshold_candidates.csv'
if validation_threshold_path.exists():
    validation_thresholds = pd.read_csv(validation_threshold_path)[[
        'threshold_name', 'precision', 'recall'
    ]].rename(columns={
        'precision': 'validation_precision',
        'recall': 'validation_recall',
    })
    cutoff_comparison = threshold_results.merge(
        validation_thresholds, on='threshold_name', how='left'
    )
    cutoff_comparison['precision_change'] = (
        cutoff_comparison['precision'] - cutoff_comparison['validation_precision']
    )
    cutoff_comparison['recall_change'] = (
        cutoff_comparison['recall'] - cutoff_comparison['validation_recall']
    )
    display(cutoff_comparison[[
        'threshold_name', 'threshold',
        'validation_precision', 'precision', 'precision_change',
        'validation_recall', 'recall', 'recall_change',
    ]])

decline_rules = rule_results['decision'].isin(['Hard Decline', 'Soft Decline'])
fraud_amount_intervened = rule_results.loc[decline_rules, 'fraud_amount'].sum()
total_fraud_amount = decision_frame.loc[
    decision_frame['is_fraud'].eq(1), 'transaction_amount'
].sum()
fraud_amount_coverage = fraud_amount_intervened / total_fraud_amount

hard_rule = rule_results.loc[
    rule_results['triggered_rule'].str.startswith('CC1.1')
].iloc[0]

specific_findings = pd.Series({
    'validation_PR_AUC': validation_pr_auc,
    'external_test_PR_AUC': test_pr_auc,
    'PR_AUC_change': test_pr_auc - validation_pr_auc,
    'external_test_ROC_AUC': model_metrics.loc[0, 'ROC-AUC'],
    'policy_decline_or_verification_rate': policy_metrics.loc[0, 'decision_rate'],
    'policy_precision': policy_metrics.loc[0, 'precision'],
    'policy_recall': policy_metrics.loc[0, 'recall'],
    'policy_false_positives': int(policy_metrics.loc[0, 'false_positives']),
    'policy_false_negatives': int(policy_metrics.loc[0, 'false_negatives']),
    'fraud_amount_intervened': fraud_amount_intervened,
    'total_test_fraud_amount': total_fraud_amount,
    'fraud_amount_coverage': fraud_amount_coverage,
    'CC1.1_hard_rule_hit_rate': hard_rule['fraud_hit_rate'],
    'CC1.1_hard_rule_fraud_coverage': hard_rule['share_of_all_fraud'],
})
display(specific_findings.to_frame('result'))

print(
    f'The frozen model achieved test PR-AUC {test_pr_auc:.4f}, '
    f'a change of {test_pr_auc - validation_pr_auc:+.4f} from validation.'
)
print(
    f'The policy intervened on {policy_metrics.loc[0, "decision_rate"]:.2%} of test '
    f'transactions, captured {policy_metrics.loc[0, "recall"]:.2%} of fraud cases, '
    f'and had {policy_metrics.loc[0, "precision"]:.2%} precision.'
)
print(
    f'It covered {fraud_amount_coverage:.2%} of observed fraud dollars before '
    'customer-confirmation and operational-cost adjustments.'
)

**What this block does:** Compares validation and external-test cutoff performance, calculates fraud-dollar coverage, and prints the exact final conclusions. It is reporting only; no model, threshold, flag, or rule is modified.

### What the final results mean

- **Model ranking weakened but remained strong.** PR-AUC decreased from 0.9145 on validation to 0.8624 on the later external test set, while test ROC-AUC was 0.9966. The later period has a lower fraud prevalence. Isolating the causes of the change requires further analysis.
- **The complete intervention policy is equivalent to `T_medium`.** Every score at or above 0.788081 receives either a hard or soft decline, so the behavioral rules decide the severity and reason rather than increasing total intervention volume. The policy intervened on 4,937 of 555,719 transactions (0.8884%).
- **Overall policy result:** 1,975 fraud cases were captured, 170 were missed, and 2,962 legitimate transactions initially received decline/verification. Precision was 40.00%, recall was 92.07%, and F1 was 55.78%.
- **`T_hard` result:** the score-only hard cutoff flagged 1,972 transactions, with 84.23% precision and 77.44% recall. This is below its validation targets of 90.03% precision and 82.18% recall.
- **`CC1.1` behavior-supported hard rule:** 1,665 transactions fired the rule; 1,451 were fraud and 214 were legitimate. Its test hit rate was 87.15%, and it captured 67.65% of all fraud cases. Behavioral evidence improved action confidence relative to score alone, but the rule still creates false positives and remains a hard-decline candidate pending cost approval.
- **Medium-score behavioral explanations varied:** interval anomaly plus risky category/time remained useful at 19.26% hit rate. New merchant plus amount spike achieved 11.51%. The history-uncertainty rule fell from 21/21 in validation to 15.33% on test, confirming that its earlier perfect result came from a very small sample and should never have been treated as a hard-decline rule.
- **Low-score approval was highly selective:** scores below `T_low` covered 532,692 transactions and contained 48 fraud cases, an observed fraud rate of approximately 0.0090%. Across all soft-approve rules, 170 fraud cases were missed.
- **Fraud-dollar coverage is an upper-bound operational estimate.** The intervention rules captured the fraud amounts associated with declined/verified transactions, but actual savings depend on customer responses, authorization timing, recoveries, review cost, and whether soft declines are ultimately released.

The frozen strategy therefore demonstrates strong coverage with a low intervention rate, but it should not be described as production-ready until the business accepts the 2,962 legitimate interventions, validates the hard-decline cost, and models customer-confirmation outcomes.

## 8. Interpretation discipline

- This is the final unbiased test evaluation for the current pipeline. Do not adjust thresholds, behavioral boundaries, selected categories, or rule order based on these results and then report the same test metrics as unbiased.
- A soft decline is treated as an initial intervention in the confusion matrix. In production, legitimate customers who successfully confirm activity may be released, so customer-friction and final-loss calculations require response and operational-cost assumptions.
- Policy, whitelist, authentication, international/currency, and confirmed-compromise rules could not be tested because those fields are absent.
- If results are unacceptable, define a new development experiment and reserve a genuinely new future-period dataset for the next final test.

## Customer history across the split
A sensitivity check replays the same scores and cutoffs after standardizing birth dates in the profile key. The comparison measures changes in how transactions are routed. It uses the same holdout and therefore serves as a retrospective diagnostic.

In [ ]:
profile_fields = ['first', 'last', 'dob', 'street']
history_fields = profile_fields + ['trans_date_trans_time', 'merchant', 'amt']

def profile_index(frame):
    return pd.MultiIndex.from_frame(frame[profile_fields].drop_duplicates())

def normalized_history_input(raw):
    data = raw[history_fields].copy()
    data['dob'] = pd.to_datetime(
        data['dob'], format='mixed', dayfirst=True, errors='raise'
    ).dt.strftime('%Y-%m-%d')
    return data

normalized_train = normalized_history_input(train_raw)
normalized_test = normalized_history_input(test_raw)
repaired_history = prepare_combined_history(normalized_train, normalized_test)
replayed = apply_policy(test_score, test_raw['amt'].to_numpy(), X_test, repaired_history)

audit_rows = []
for version, train_input, test_input, frame in [
    ('Original text key', train_raw, test_raw, decision_frame),
    ('Normalized birth date', normalized_train, normalized_test, replayed),
]:
    test_profiles = profile_index(test_input)
    overlap = profile_index(train_input).intersection(test_profiles)
    hard = frame['decision'].eq('Hard Decline')
    soft = frame['decision'].eq('Soft Decline')
    audit_rows.append({
        'profile_key': version,
        'test_profiles': len(test_profiles),
        'matched_training_profiles': len(overlap),
        'hard_declines': int(hard.sum()),
        'fraud_hard_declined': int(y_test[hard].sum()),
        'legitimate_hard_declined': int(hard.sum() - y_test[hard].sum()),
        'verification_requests': int(soft.sum()),
        'interventions': int((hard | soft).sum()),
        'action_changed_vs_original': int(frame['decision'].ne(decision_frame['decision']).sum()),
        'first_hit_rule_changed_vs_original': int(frame['triggered_rule'].ne(decision_frame['triggered_rule']).sum()),
    })
    assert np.array_equal((hard | soft).to_numpy(), test_score >= T_MEDIUM)

audit_results = pd.DataFrame(audit_rows)
audit_results.to_csv(OUTPUT_DIR / 'history_routing_audit.csv', index=False)
display(audit_results)
